In [1]:
!apt-get install tesseract-ocr -y
!apt-get install poppler-utils -y

!pip install pymupdf
!pip install pdf2image
!pip install pytesseract
!pip install python-docx

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (2,415 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Process

In [2]:
from google.colab import files

uploaded = files.upload()

Saving 0001_INV-273789.pdf to 0001_INV-273789.pdf


In [3]:
import fitz
import pytesseract
from pdf2image import convert_from_path
from docx import Document
import matplotlib.pyplot as plt

In [4]:
file_name = list(uploaded.keys())[0]

print("Uploaded File:", file_name)

Uploaded File: 0001_INV-273789.pdf


In [5]:
def extract_text(file_name):

    text = ""

    # DOCX FILE
    if file_name.endswith(".docx"):

        doc = Document(file_name)

        text = "\n".join(
            [para.text for para in doc.paragraphs]
        )

        return text

    # PDF FILE
    elif file_name.endswith(".pdf"):

        pdf = fitz.open(file_name)

        for page in pdf:
            text += page.get_text()

        # If text exists, return it
        if len(text.strip()) > 50:
            print("Text-based PDF detected")
            return text

        print("Scanned PDF detected. Applying OCR...")

        text = ""

        images = convert_from_path(file_name)

        for img in images:
            text += pytesseract.image_to_string(img)

        return text

    else:
        raise ValueError(
            "Only PDF and DOCX files are supported"
        )

In [6]:
text = extract_text(file_name)

print(text)

Scanned PDF detected. Applying OCR...
 

BlueNet Communications Ltd.
House 12, Road 8, Banani, Dhaka

 

TAX INVOICE
Account: 1.62687160
= BIN: 291237044-6823
Invoice: INV-273789
Date: November 25, 2024
Due: December 02, 2024
| Received From / Client

 
  
 

Delta Foods Bangladesh
Industrial Area, Narayanganj
Contact: Sabbir Anmed

 
 
 

 

     
   
   
 

Payment Reference
Bill Period: 26-Oct-2024 to 24-Nov-2024 |
Mobile/Ref: +8801644445555

Currency. BDT |

a | cami sencetncion | ay
2 [enwatrening —SSSCS*~S~S~Sa TS
i 0

   
   
      

 

Customer Copy

Name: Delta Foods Bangladesh
Invoice Number: INV-273789
Date: November 25, 2024
Account No: 1.62687 160

Total Amount Due: 153,555.50
Last Date: December 02, 2024

Bank Copy

Name: Delta Foods Bangladesh
Invoice Number: INV-273789
Date: November 25, 2024
Account No: 1.62687160

Total Amount Due: 153,555.50
Last Date: December 02, 2024

Payment should be posted against the invoice/account reference shown above.



In [7]:
import re

In [8]:
def clean_text(text):

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove special characters except numbers, letters, ₹, /, -, :
    text = re.sub(r'[^A-Za-z0-9₹:/\- ]', ' ', text)

    # Remove multiple spaces again
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [9]:
cleaned_text = clean_text(text)

print(cleaned_text[:2000])

BlueNet Communications Ltd House 12 Road 8 Banani Dhaka TAX INVOICE Account: 1 62687160 BIN: 291237044-6823 Invoice: INV-273789 Date: November 25 2024 Due: December 02 2024 Received From / Client Delta Foods Bangladesh Industrial Area Narayanganj Contact: Sabbir Anmed Payment Reference Bill Period: 26-Oct-2024 to 24-Nov-2024 Mobile/Ref: 8801644445555 Currency BDT a cami sencetncion ay 2 enwatrening SSSCS S S Sa TS i 0 Customer Copy Name: Delta Foods Bangladesh Invoice Number: INV-273789 Date: November 25 2024 Account No: 1 62687 160 Total Amount Due: 153 555 50 Last Date: December 02 2024 Bank Copy Name: Delta Foods Bangladesh Invoice Number: INV-273789 Date: November 25 2024 Account No: 1 62687160 Total Amount Due: 153 555 50 Last Date: December 02 2024 Payment should be posted against the invoice/account reference shown above


In [14]:
cleaned_text = cleaned_text.lower()

print(cleaned_text[:5000])

bluenet communications ltd house 12 road 8 banani dhaka tax invoice account: 1 62687160 bin: 291237044-6823 invoice: inv-273789 date: november 25 2024 due: december 02 2024 received from / client delta foods bangladesh industrial area narayanganj contact: sabbir anmed payment reference bill period: 26-oct-2024 to 24-nov-2024 mobile/ref: 8801644445555 currency bdt a cami sencetncion ay 2 enwatrening ssscs s s sa ts i 0 customer copy name: delta foods bangladesh invoice number: inv-273789 date: november 25 2024 account no: 1 62687 160 total amount due: 153 555 50 last date: december 02 2024 bank copy name: delta foods bangladesh invoice number: inv-273789 date: november 25 2024 account no: 1 62687160 total amount due: 153 555 50 last date: december 02 2024 payment should be posted against the invoice/account reference shown above


In [11]:
import re

In [17]:
def extract_contract_details(text):

    result = {}

    supplier = re.search(
        r'supplier\s*:?\s*([a-zA-Z0-9 &.,]+)',
        text,
        re.IGNORECASE
    )

    product = re.search(
        r'product\s*:?\s*([a-zA-Z0-9 &.,]+)',
        text,
        re.IGNORECASE
    )

    quantity = re.search(
        r'quantity\s*:?\s*([a-zA-Z0-9 ]+)',
        text,
        re.IGNORECASE
    )

    price = re.search(
        r'price\s*:?\s*₹?\s*([0-9,]+)',
        text,
        re.IGNORECASE
    )

    delivery_date = re.search(
        r'delivery\s*date\s*:?\s*([a-zA-Z0-9 ,]+)',
        text,
        re.IGNORECASE
    )

    payment_terms = re.search(
        r'payment\s*terms\s*:?\s*([a-zA-Z0-9 ]+)',
        text,
        re.IGNORECASE
    )

    result["supplier"] = supplier.group(1).strip() if supplier else None
    result["product"] = product.group(1).strip() if product else None
    result["quantity"] = quantity.group(1).strip() if quantity else None
    result["price"] = price.group(1).strip() if price else None
    result["delivery_date"] = delivery_date.group(1).strip() if delivery_date else None
    result["payment_terms"] = payment_terms.group(1).strip() if payment_terms else None

    return result

In [19]:
invoice_info = extract_contract_details(cleaned_text)

invoice_info

{'supplier': None,
 'product': None,
 'quantity': None,
 'price': None,
 'delivery_date': None,
 'payment_terms': None}

In [20]:
!pip install transformers
!pip install sentencepiece
!pip install torch

In [23]:
!pip install transformers sentencepiece accelerate -q

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [25]:
contract_text = """
ABC Chemicals agrees to supply 5000 Tons of Industrial Solvent at ₹35000 per Ton.

Delivery shall be completed by 15 July 2026.

Payment will be made within 30 days after successful delivery.

The supplier shall ensure quality standards and comply with all safety regulations.
"""

prompt = f"""
Summarize the following procurement contract:

{contract_text}
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(summary)

Purchase 5000 Tons of Industrial Solvent from ABC Chemicals.


In [26]:
import transformers

print(transformers.__version__)

5.9.0


In [27]:
!pip install sentencepiece accelerate -q

In [28]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [29]:
contract_text = """
ABC Chemicals agrees to supply 5000 Tons of Industrial Solvent at ₹35000 per Ton.

Delivery shall be completed by 15 July 2026.

Payment will be made within 30 days after successful delivery.

The supplier shall ensure quality standards and comply with all safety regulations.
"""

input_text = "summarize: " + contract_text

inputs = tokenizer.encode(
    input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

summary_ids = model.generate(
    inputs,
    max_length=80,
    min_length=20,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print(summary)

ABC Chemicals agrees to supply 5000 Tons of Industrial Solvent at 35000 per Ton. payment will be made within 30 days after successful delivery.


In [30]:
final_result = contract_info.copy()

final_result["summary"] = summary

final_result

{'supplier': None,
 'product': None,
 'quantity': None,
 'price': None,
 'delivery_date': None,
 'payment_terms': None,
 'summary': 'ABC Chemicals agrees to supply 5000 Tons of Industrial Solvent at 35000 per Ton. payment will be made within 30 days after successful delivery.'}

In [31]:
!pip install -U spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 40.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [32]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [33]:
doc = nlp(contract_text)

for ent in doc.ents:
    print(f"{ent.text:<30} ---> {ent.label_}")

ABC Chemicals                  ---> ORG
5000 Tons                      ---> QUANTITY
35000                          ---> MONEY
15 July 2026                   ---> DATE
30 days                        ---> DATE


In [34]:
doc = nlp(contract_text)

for ent in doc.ents:
    print(f"{ent.text:<30} ---> {ent.label_}")

ABC Chemicals                  ---> ORG
5000 Tons                      ---> QUANTITY
35000                          ---> MONEY
15 July 2026                   ---> DATE
30 days                        ---> DATE


In [35]:
import pandas as pd

entities = []

for ent in doc.ents:
    entities.append(
        {
            "Entity": ent.text,
            "Label": ent.label_
        }
    )

df = pd.DataFrame(entities)

df

,Entity,Label
0,ABC Chemicals,ORG
1,5000 Tons,QUANTITY
2,35000,MONEY
3,15 July 2026,DATE
4,30 days,DATE


In [36]:
procurement_data = {
    "supplier": None,
    "quantity": None,
    "price": None,
    "delivery_date": None
}

for ent in doc.ents:

    if ent.label_ == "ORG":
        procurement_data["supplier"] = ent.text

    elif ent.label_ == "QUANTITY":
        procurement_data["quantity"] = ent.text

    elif ent.label_ == "MONEY":
        procurement_data["price"] = ent.text

    elif ent.label_ == "DATE":

        if procurement_data["delivery_date"] is None:
            procurement_data["delivery_date"] = ent.text

procurement_data

{'supplier': 'ABC Chemicals',
 'quantity': '5000 Tons',
 'price': '35000',
 'delivery_date': '15 July 2026'}

In [37]:
TRAIN_DATA = [
    (
        "Supplier ABC Chemicals will deliver 5000 Tons by 15 July 2026.",
        {
            "entities": [
                (9, 22, "SUPPLIER"),
                (36, 45, "QUANTITY"),
                (49, 61, "DELIVERY_DATE")
            ]
        }
    )
]

In [38]:
import pandas as pd
import random

In [39]:
suppliers = [
    "ABC Chemicals",
    "Bharat Steel Ltd",
    "Green Agro Pvt Ltd",
    "National Plastics",
    "Global Minerals"
]

products = [
    "Industrial Solvent",
    "Iron Rods",
    "Fertilizer",
    "Plastic Granules",
    "Copper Wire"
]

In [40]:
import pandas as pd
import random

suppliers = [
    "ABC Chemicals",
    "Bharat Steel Ltd",
    "Green Agro Pvt Ltd",
    "National Plastics",
    "Global Minerals"
]

products = [
    "Industrial Solvent",
    "Iron Rods",
    "Fertilizer",
    "Plastic Granules",
    "Copper Wire"
]

payment_terms = [
    "Net 30 Days",
    "Net 45 Days",
    "Advance Payment",
    "50% Advance",
    "Payment After Delivery"
]

contracts = []

for i in range(500):

    supplier = random.choice(suppliers)
    product = random.choice(products)

    quantity = random.randint(100, 10000)
    price = random.randint(500, 50000)

    day = random.randint(1, 28)
    month = random.choice([
        "January",
        "February",
        "March",
        "April",
        "May",
        "June",
        "July",
        "August",
        "September",
        "October",
        "November",
        "December"
    ])

    delivery_date = f"{day} {month} 2026"

    payment = random.choice(payment_terms)

    contract_text = f"""
    Supplier: {supplier}

    Product: {product}

    Quantity: {quantity} Tons

    Price: ₹{price}/Ton

    Delivery Date: {delivery_date}

    Payment Terms: {payment}

    {supplier} agrees to supply {quantity} Tons of {product}
    at ₹{price} per Ton.

    Delivery shall be completed by {delivery_date}.

    Payment Terms: {payment}.
    """

    contracts.append(
        {
            "contract_text": contract_text,
            "supplier": supplier,
            "product": product,
            "quantity": f"{quantity} Tons",
            "price": str(price),
            "delivery_date": delivery_date,
            "payment_terms": payment
        }
    )

df = pd.DataFrame(contracts)

df.head()

,contract_text,supplier,product,quantity,price,delivery_date,payment_terms
0,\n Supplier: National Plastics\n\n Produ...,National Plastics,Plastic Granules,2017 Tons,42806,8 October 2026,Payment After Delivery
1,\n Supplier: Global Minerals\n\n Product...,Global Minerals,Plastic Granules,446 Tons,28742,14 February 2026,50% Advance
2,\n Supplier: National Plastics\n\n Produ...,National Plastics,Industrial Solvent,3533 Tons,18526,12 March 2026,50% Advance
3,\n Supplier: Bharat Steel Ltd\n\n Produc...,Bharat Steel Ltd,Iron Rods,9587 Tons,37725,3 August 2026,Payment After Delivery
4,\n Supplier: ABC Chemicals\n\n Product: ...,ABC Chemicals,Plastic Granules,2758 Tons,26367,25 February 2026,Net 30 Days


In [41]:
df.to_csv(
    "procurement_contracts.csv",
    index=False
)

print("Dataset Saved Successfully")

Dataset Saved Successfully


In [42]:
import re
import fitz
import pytesseract
import spacy

from pdf2image import convert_from_path
from docx import Document

from transformers import T5Tokenizer, T5ForConditionalGeneration


# =========================
# LOAD MODELS ONCE
# =========================

nlp = spacy.load("en_core_web_sm")

tokenizer = T5Tokenizer.from_pretrained("t5-small")
summarizer_model = T5ForConditionalGeneration.from_pretrained("t5-small")


# =========================
# READ DOCUMENT
# =========================

def extract_text(file_path):

    text = ""

    if file_path.endswith(".docx"):

        doc = Document(file_path)

        text = "\n".join(
            [para.text for para in doc.paragraphs]
        )

        return text

    elif file_path.endswith(".pdf"):

        pdf = fitz.open(file_path)

        for page in pdf:
            text += page.get_text()

        if len(text.strip()) > 50:
            return text

        images = convert_from_path(file_path)

        text = ""

        for img in images:
            text += pytesseract.image_to_string(img)

        return text

    else:
        raise ValueError(
            "Only PDF and DOCX files are supported"
        )


# =========================
# CLEAN TEXT
# =========================

def clean_text(text):

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(
        r'[^A-Za-z0-9₹:/\-., ]',
        ' ',
        text
    )

    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# =========================
# NER EXTRACTION
# =========================

def extract_entities(text):

    doc = nlp(text)

    result = {
        "supplier": None,
        "quantity": None,
        "price": None,
        "delivery_date": None
    }

    for ent in doc.ents:

        if ent.label_ == "ORG":

            if result["supplier"] is None:
                result["supplier"] = ent.text

        elif ent.label_ == "QUANTITY":

            if result["quantity"] is None:
                result["quantity"] = ent.text

        elif ent.label_ == "MONEY":

            if result["price"] is None:
                result["price"] = ent.text

        elif ent.label_ == "DATE":

            if result["delivery_date"] is None:
                result["delivery_date"] = ent.text

    return result


# =========================
# REGEX EXTRACTION
# =========================

def extract_contract_fields(text):

    supplier = re.search(
        r'Supplier\s*:?\s*(.*?)\s*Product',
        text,
        re.IGNORECASE
    )

    product = re.search(
        r'Product\s*:?\s*(.*?)\s*Quantity',
        text,
        re.IGNORECASE
    )

    quantity = re.search(
        r'Quantity\s*:?\s*(.*?)\s*Price',
        text,
        re.IGNORECASE
    )

    price = re.search(
        r'Price\s*:?\s*₹?([0-9,]+)',
        text,
        re.IGNORECASE
    )

    delivery_date = re.search(
        r'Delivery Date\s*:?\s*(.*?)\s*Payment',
        text,
        re.IGNORECASE
    )

    payment_terms = re.search(
        r'Payment Terms\s*:?\s*(.*)',
        text,
        re.IGNORECASE
    )

    return {

        "supplier":
        supplier.group(1).strip()
        if supplier else None,

        "product":
        product.group(1).strip()
        if product else None,

        "quantity":
        quantity.group(1).strip()
        if quantity else None,

        "price":
        price.group(1).strip()
        if price else None,

        "delivery_date":
        delivery_date.group(1).strip()
        if delivery_date else None,

        "payment_terms":
        payment_terms.group(1).strip()
        if payment_terms else None
    }


# =========================
# SUMMARIZATION
# =========================

def generate_summary(text):

    input_text = "summarize: " + text

    inputs = tokenizer.encode(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    summary_ids = summarizer_model.generate(
        inputs,
        max_length=100,
        min_length=20,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary


# =========================
# MASTER FUNCTION
# =========================

def analyze_contract(file_path):

    raw_text = extract_text(file_path)

    cleaned_text = clean_text(raw_text)

    regex_data = extract_contract_fields(
        cleaned_text
    )

    ner_data = extract_entities(
        cleaned_text
    )

    summary = generate_summary(
        cleaned_text
    )

    result = {

        "supplier":
        regex_data["supplier"]
        or ner_data["supplier"],

        "product":
        regex_data["product"],

        "quantity":
        regex_data["quantity"]
        or ner_data["quantity"],

        "price":
        regex_data["price"]
        or ner_data["price"],

        "delivery_date":
        regex_data["delivery_date"]
        or ner_data["delivery_date"],

        "payment_terms":
        regex_data["payment_terms"],

        "summary":
        summary
    }

    return result

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]